# 04 — Cascade: making legal-bert fast enough for a free CPU

**Problem:** on Cloud Run's single vCPU, legal-bert takes ~280 ms per sentence. A long ToS (TikTok: 700+ sentences) took over 2 minutes and timed out.

**Idea:** ~89% of ToS sentences are fair. The TF-IDF baseline is ~7,000× faster, so let it **screen** every sentence and send only the "maybe unfair" ones (max probability ≥ gate) to legal-bert. Everything else is treated as fair.

**Protocol:** choose the gate on **validation**, confirm once on **test**.

In [1]:
import time

import joblib
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from clausula.data import load_splits
from clausula.labels import LABELS
from clausula.metrics import summary

splits = load_splits()
baseline = joblib.load("../models/baseline.joblib")
BERT = "../models/legal-bert-base-uncased"
tok = AutoTokenizer.from_pretrained(BERT)
bert = AutoModelForSequenceClassification.from_pretrained(BERT).eval()
thresholds = np.load(f"{BERT}/thresholds.npy")
device = "cuda" if torch.cuda.is_available() else "cpu"
bert.to(device)

def bert_probs(texts):
    out = []
    for i in range(0, len(texts), 64):
        enc = tok(texts[i:i + 64], truncation=True, max_length=256, padding=True, return_tensors="pt").to(device)
        with torch.inference_mode():
            out.append(torch.sigmoid(bert(**enc).logits).float().cpu().numpy())
    return np.concatenate(out)

probs = {name: {"baseline": baseline["model"].predict_proba(df["text"]), "bert": bert_probs(df["text"].tolist())}
         for name, df in splits.items() if name in ("validation", "test")}

/home/davig/clausula/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2322.35it/s]

## 1. Choose the gate on validation

In [2]:
def cascade_table(split):
    y = splits[split][LABELS].values
    pb, pt = probs[split]["baseline"], probs[split]["bert"]
    rows = [{"gate": "none (bert only)", "sent to bert": 1.0, "macro_f1": summary(y, pt >= thresholds)["macro_f1"], "unfair skipped": 0}]
    for gate in [0.02, 0.05, 0.1, 0.2, 0.3]:
        send = pb.max(axis=1) >= gate
        pred = np.where(send[:, None], pt >= thresholds, False)
        rows.append({"gate": gate, "sent to bert": send.mean(), "macro_f1": summary(y, pred)["macro_f1"],
                     "unfair skipped": int((y.any(axis=1) & ~send).sum())})
    return pd.DataFrame(rows).round(3)

cascade_table("validation")

,gate,sent to bert,macro_f1,unfair skipped
0,none (bert only),1.000,0.779,0
1,0.02,0.385,0.783,5
2,0.05,0.253,0.783,10
3,0.1,0.188,0.769,17
4,0.2,0.149,0.765,25
5,0.3,0.133,0.757,32


**Gate = 0.05**: legal-bert reads only ~25% of sentences, macro-F1 is unchanged (it even goes up slightly, because the baseline filters some of legal-bert's false positives). Higher gates start skipping too many unfair clauses.

## 2. Confirm on test (once)

In [3]:
cascade_table("test")

,gate,sent to bert,macro_f1,unfair skipped
0,none (bert only),1.000,0.780,0
1,0.02,0.360,0.781,3
2,0.05,0.241,0.785,7
3,0.1,0.187,0.794,12
4,0.2,0.146,0.787,17
5,0.3,0.126,0.783,25


## 3. Speed on a single CPU thread (like Cloud Run)

In [4]:
torch.set_num_threads(1)
bert.to("cpu")
device = "cpu"
sample = splits["test"]["text"].tolist()[:300]

t = time.perf_counter(); bert_probs(sample); full = (time.perf_counter() - t) / len(sample) * 1000
send = baseline["model"].predict_proba(sample).max(axis=1) >= 0.05
t = time.perf_counter(); bert_probs([s for s, k in zip(sample, send, strict=True) if k]); casc = (time.perf_counter() - t) / len(sample) * 1000
print(f"legal-bert only: {full:.0f} ms/sentence | cascade: {casc:.0f} ms/sentence ({full / casc:.1f}x faster)")

legal-bert only: 379 ms/sentence | cascade: 120 ms/sentence (3.2x faster)


## Conclusion

- **Quality is kept:** with gate 0.05 (chosen on validation), test macro-F1 is 0.785 vs 0.780 for legal-bert alone, while legal-bert reads only ~1 in 4 sentences.
- **Speed:** the cascade alone is ~3× faster on one CPU thread (cell above, plain batches). `TransformerDetector` also batches sentences of similar length together (less padding, ~2× faster on its own). End to end, `analyze()` on a 1,800-sentence document went from ~280 to **~28 ms per sentence** on a single CPU: a 700-sentence ToS takes ~20 s instead of timing out.
